# Singular Value Decomposition (SVD)

## Definition
SVD factorizes **any** matrix A (m × n) into three matrices:

```
A = U @ Σ @ V^T
```

where:
- **U** is an (m × m) orthogonal matrix — left singular vectors
- **Σ** (Sigma) is an (m × n) diagonal matrix — singular values (non-negative, sorted descending)
- **V^T** is an (n × n) orthogonal matrix — right singular vectors (rows = right singular vectors)

### Properties
- Singular values σ₁ ≥ σ₂ ≥ ... ≥ σₖ ≥ 0 (where k = min(m, n))
- Works on **any** matrix (square, rectangular, singular)
- `np.linalg.svd(A, full_matrices=True/False)`
- `full_matrices=False` → economy/thin SVD, more memory efficient

### AI/ML Applications
| Use Case | How SVD is used |
|----------|----------------|
| **PCA** | SVD of data matrix = principal components |
| **Recommender Systems** | Matrix factorization (collaborative filtering) |
| **NLP (LSA)** | SVD of term-document matrix |
| **Image Compression** | Low-rank approximation |
| **Pseudo-inverse** | `A⁺ = V @ Σ⁺ @ U^T` |
| **Condition number** | σ_max / σ_min |

In [ ]:
import numpy as np

# Basic SVD
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9],
              [10, 11, 12]], dtype=float)  # (4, 3)

U, s, Vt = np.linalg.svd(A, full_matrices=False)

print('A shape:  ', A.shape)    # (4, 3)
print('U shape:  ', U.shape)    # (4, 3) -- columns are left singular vectors
print('s shape:  ', s.shape)    # (3,)   -- singular values
print('Vt shape: ', Vt.shape)   # (3, 3) -- rows are right singular vectors
print()
print('Singular values:', np.round(s, 4))

In [ ]:
# Verify reconstruction: A = U @ diag(s) @ Vt
U, s, Vt = np.linalg.svd(A, full_matrices=False)
A_reconstructed = U @ np.diag(s) @ Vt
print('Reconstruction error:', np.linalg.norm(A - A_reconstructed))  # ~0

In [ ]:
# Low-rank approximation (image compression simulation)
rng = np.random.default_rng(42)
A = rng.standard_normal((100, 100))  # "image"

U, s, Vt = np.linalg.svd(A, full_matrices=False)

for k in [1, 5, 10, 20, 50]:
    # Keep only top-k singular values
    A_approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    error = np.linalg.norm(A - A_approx, 'fro') / np.linalg.norm(A, 'fro')
    compression = k * (100 + 1 + 100) / (100 * 100)
    print(f'k={k:3d}: relative error={error:.4f}, compression ratio={compression:.3f}')

In [ ]:
# Explained variance ratio (like PCA with SVD)
rng = np.random.default_rng(42)
X = rng.standard_normal((200, 10))  # (n_samples, n_features)
X_centered = X - X.mean(axis=0)

_, s, Vt = np.linalg.svd(X_centered, full_matrices=False)

# Singular values squared / (n-1) = eigenvalues of covariance
explained = s**2 / (len(X) - 1)
explained_ratio = explained / explained.sum()
cumulative = np.cumsum(explained_ratio)

print('Explained variance ratio:', np.round(explained_ratio, 4))
print('Cumulative:', np.round(cumulative, 4))

In [ ]:
# PCA using SVD (numerically preferred over eigendecomposition)
rng = np.random.default_rng(42)
X = rng.standard_normal((500, 20))

# Center
X_c = X - X.mean(axis=0)

# SVD
U, s, Vt = np.linalg.svd(X_c, full_matrices=False)

# Principal components (right singular vectors = rows of Vt)
components = Vt  # (20, 20)

# Project to top-2 dimensions
X_pca = X_c @ components[:2].T   # (500, 2)
print('PCA projected shape:', X_pca.shape)

# Equivalent: X_c @ Vt.T[:, :2] or U[:, :2] * s[:2]
X_pca2 = U[:, :2] * s[:2]
print('Equivalent match:', np.allclose(np.abs(X_pca), np.abs(X_pca2)))

In [ ]:
# Recommender system: Matrix factorization via SVD
# Rating matrix (users x movies), 0 = unrated
ratings = np.array([
    [5, 3, 0, 1],
    [4, 0, 0, 1],
    [1, 1, 0, 5],
    [1, 0, 0, 4],
    [0, 1, 5, 4],
], dtype=float)

U, s, Vt = np.linalg.svd(ratings, full_matrices=False)

# Low-rank approximation with k=2
k = 2
R_approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
print('Original ratings:\n', ratings)
print('Low-rank approximation (k=2):\n', np.round(R_approx, 2))

In [ ]:
# Compute pseudo-inverse via SVD
# A+ = V @ Sigma+ @ U^T
A = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)
U, s, Vt = np.linalg.svd(A, full_matrices=False)

# Reciprocal of non-zero singular values
s_inv = 1.0 / s  # threshold small values in practice
A_pinv_manual = Vt.T @ np.diag(s_inv) @ U.T

A_pinv_np = np.linalg.pinv(A)
print('Manual pinv match:', np.allclose(A_pinv_manual, A_pinv_np))
print('Pseudo-inverse:\n', np.round(A_pinv_np, 4))